# Data Exploration and Preprocessing

In [ ]:
import pandas as pd
import duckdb
import os
import sys
import spacy

sys.path.append(os.path.abspath('..'))

from src.utils import make_corpus, preprocess_spacy

## EDA

In [2]:
# Connect to DuckDB
c2 = duckdb.connect()

In [3]:
# Loading the data and view the example records
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,4.0,It holds the water and makes bubbles. That's ...,"It's cheap and it does what I wanted. The ""ma...",[],B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,1662258542725,7,True,"Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...",NaN,4.4,Health & Personal Care,Homedics
1,1.0,Not for me,Didn't do a thing for me. Not saying they don'...,[],B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,1642722787262,0,True,Brain Supplement 1053mg - Premium Nootropic Br...,NaN,4.1,Health & Personal Care,Nature's Nutrition
2,4.0,Makes a nice compact noise machine to take wit...,This is a nice basic sound machine. I have use...,[],B08THJD1MH,B08THJD1MH,AFSKPY37N3C43SOI5IEXEK5JSIYA,1617907534645,0,False,White Air Purifier and Dehumidifier Q10 True H...,NaN,2.8,Health & Personal Care,Afloia
3,5.0,Great hair dryer gets the job done quickly!,This Jinri hair dryer is among one of the best...,[],B0895LW9LL,B0895LW9LL,AFSKPY37N3C43SOI5IEXEK5JSIYA,1601757145389,0,False,"Jinri Professional Tourmaline Hair Dryer, Nega...",99.99,4.3,Health & Personal Care,JINRI
4,5.0,Great reasonably priced shower filter!,I live in Florida where hard water is definite...,[],B07PKLN99K,B07PKLN99K,AFSKPY37N3C43SOI5IEXEK5JSIYA,1559421381258,0,False,Gophra 15 Stage Shower Filter with 1 Cartridge...,NaN,4.5,Health & Personal Care,Gophra


In [4]:
# Summary of the numerical columns
data.describe()

,rating,timestamp,helpful_vote,price,average_rating
count,20000.00000,2.000000e+04,20000.000000,3994.000000,9006.000000
mean,4.13810,1.543728e+12,1.419650,27.779850,4.175205
std,1.32884,8.079927e+10,11.843794,29.955227,0.517167
min,1.00000,1.024047e+12,0.000000,1.020000,1.000000
25%,4.00000,1.485797e+12,0.000000,11.990000,3.900000
50%,5.00000,1.556722e+12,0.000000,19.990000,4.300000
75%,5.00000,1.607991e+12,1.000000,32.950000,4.500000
max,5.00000,1.679261e+12,737.000000,650.000000,5.000000


In [5]:
# View the dataset size
data.shape

(20000, 15)

In [6]:
# View the dataset fields and the column types
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   rating             20000 non-null  float64
 1   title              20000 non-null  str    
 2   text               20000 non-null  str    
 3   images             20000 non-null  object 
 4   asin               20000 non-null  str    
 5   parent_asin        20000 non-null  str    
 6   user_id            20000 non-null  str    
 7   timestamp          20000 non-null  int64  
 8   helpful_vote       20000 non-null  int64  
 9   verified_purchase  20000 non-null  bool   
 10  product_title      9006 non-null   str    
 11  price              3994 non-null   float64
 12  average_rating     9006 non-null   float64
 13  main_category      9006 non-null   str    
 14  store              8822 non-null   str    
dtypes: bool(1), float64(3), int64(2), object(1), str(8)
memory usage: 9.9+ MB


## Cleaning the Data

In [7]:
# dropping all missing values in product title since missing product title would be less useful for our query
processed_df = data.copy().dropna(subset=['product_title'])
processed_df.head()

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,product_title,price,average_rating,main_category,store
0,4.0,It holds the water and makes bubbles. That's ...,"It's cheap and it does what I wanted. The ""ma...",[],B007HY7GC2,B092RP73CX,AEZGPLOYTSAPR3DHZKKXEFPAXUAA,1662258542725,7,True,"Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma...",NaN,4.4,Health & Personal Care,Homedics
1,1.0,Not for me,Didn't do a thing for me. Not saying they don'...,[],B08KYJLF5T,B08KYJLF5T,AEQAYV7RXZEBXMQIQPL6KCT2CFWQ,1642722787262,0,True,Brain Supplement 1053mg - Premium Nootropic Br...,NaN,4.1,Health & Personal Care,Nature's Nutrition
2,4.0,Makes a nice compact noise machine to take wit...,This is a nice basic sound machine. I have use...,[],B08THJD1MH,B08THJD1MH,AFSKPY37N3C43SOI5IEXEK5JSIYA,1617907534645,0,False,White Air Purifier and Dehumidifier Q10 True H...,NaN,2.8,Health & Personal Care,Afloia
3,5.0,Great hair dryer gets the job done quickly!,This Jinri hair dryer is among one of the best...,[],B0895LW9LL,B0895LW9LL,AFSKPY37N3C43SOI5IEXEK5JSIYA,1601757145389,0,False,"Jinri Professional Tourmaline Hair Dryer, Nega...",99.99,4.3,Health & Personal Care,JINRI
4,5.0,Great reasonably priced shower filter!,I live in Florida where hard water is definite...,[],B07PKLN99K,B07PKLN99K,AFSKPY37N3C43SOI5IEXEK5JSIYA,1559421381258,0,False,Gophra 15 Stage Shower Filter with 1 Cartridge...,NaN,4.5,Health & Personal Care,Gophra


In [8]:
processed_df.shape

(9006, 15)

## Make the Corpus and Pre-processing

We choose to include: (review) title, text, product_title, main_category, and
store to join in the corpus text.

The reasons: 
- product titles are main identifier for the product
- main category and which store the items belong to could be useful product features
- reviews text are helpful as futher description for our products, may be useful for semantic query

In [10]:
# extracting what fields we want
cols = ['product_title', 'main_category', 'store', 'title', 'text']

# make corpus
corpus = make_corpus(df=processed_df, cols=cols, asin="asin")
corpus.head()

,asin,text
0,B007HY7GC2,"Homedics Bubble Bliss Deluxe-Foot Spa, Heat Ma..."
1,B08KYJLF5T,Brain Supplement 1053mg - Premium Nootropic Br...
2,B08THJD1MH,White Air Purifier and Dehumidifier Q10 True H...
3,B0895LW9LL,"Jinri Professional Tourmaline Hair Dryer, Nega..."
4,B07PKLN99K,Gophra 15 Stage Shower Filter with 1 Cartridge...


In [11]:
corpus.shape

(9006, 2)

### Preprocessing Steps

1. Start with tokenization, where we split the text into individual tokens/words

2. Check each token/word:
    - Remove stop words (e.g. "a", "the", "is", etc.)
    - Remove tokens that are shorter than the minimum threshold (we use two characters length)
    - Remove tokens that include in irrelevant Part of Speech (POS). We include: adverd, pronouns, coordinating conjuction, punctuation, particle, determiner, preposition, symbol and numbers
    - Remove words with no vector (out of vocabulary)
    - Make sure the token is only made up only alphabetic characters

3. Proceed to lemmatization, reducing each word into its base/root form. (e.g. running -> run, beautifully -> beautiful, etc.)

4. Transform the final text to all lower case for uniformity

5. Combine all of the processed tokens with space as the separator

In [12]:
# begin preprocessing corpus
nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])
corpus["text"] = [preprocess_spacy(text) for text in nlp.pipe(corpus["text"])]

In [13]:
corpus.head()

,asin,text
0,B007HY7GC2,bubble bliss deluxe foot spa heat maintenance ...
1,B08KYJLF5T,brain supplement premium nootropic brain suppo...
2,B08THJD1MH,white air purifier dehumidifier true hepa air ...
3,B0895LW9LL,professional tourmaline hair dryer negative io...
4,B07PKLN99K,stage shower filter cartridge replacement remo...
